# 청년 복지정책 RAG 테스트셋 생성 및 평가 파이프라인 (단일쿼리)

## 개요
복지 정책 원본 데이터에서 청년 관련 정책만 필터링한 뒤, `ragas` 라이브러리로 Knowledge Graph를 구성하고 페르소나 기반 Single-hop(단일 문서 기반) 질문/답변 테스트셋을 생성 · 평가하는 파이프라인입니다. 정책 필터링 → 문서 변환 → Knowledge Graph 생성 → 페르소나 정의 → Single-hop 시나리오/질문 생성 → 테스트셋 생성 → RAGAS 평가 순서로 진행됩니다. 각 코드 셀 위에는 번호와 함께 어떤 단계인지 짧게 표시해두었습니다.

> **저작권 안내**: 이 코드가 접근하는 실제 사이트 도메인은 저작권 문제로 `target_site`라는 이름으로 치환하였습니다.

## API 키 설정
코드 곳곳에서 사용하는 `api_key` 변수(OpenAI API 키)는 보안을 위해 실제 키 값을 코드에서 제거했습니다. 실행하려면 아래처럼 본인의 환경변수에서 값을 불러오는 코드를 직접 추가해야 합니다.
```python
import os
api_key = os.environ["OPENAI_API_KEY"]
```
이 코드는 `api_key`를 처음 사용하는 셀(LLM·임베딩 초기화) 이전에 실행되어야 합니다.

## 주요 기능
1. **청년 정책 필터링** — 키워드/연령/위치 기반 점수화로 전체 정책 중 청년 관련 정책만 선별합니다.
2. **Knowledge Graph 구성** — 필터링된 정책 문서를 `ragas`의 `KnowledgeGraph`로 변환하고, Headline/Keyphrase 추출과 문서 간 연관관계(overlap) 계산을 적용합니다.
3. **페르소나 정의** — 취업준비생, 사회초년생, 신혼부부 등 청년 유형별 페르소나를 정의합니다.
4. **Single-hop 질문 생성** — 하나의 정책 문서 내용만으로 답할 수 있는 질문을 커스텀 `SingleHopQuerySynthesizer`로 생성합니다.
5. **테스트셋 생성 및 저장** — `TestsetGenerator`로 질문·정답·근거를 포함한 테스트셋을 만들고 JSON으로 저장합니다.
6. **RAGAS 평가** — Faithfulness / AnswerRelevancy / ContextPrecision / ContextRecall 지표와 Acceptable Threshold 기준으로 생성된 테스트셋의 품질을 평가합니다.

## 코드 구조
- **청년 정책 필터링** — 키워드 기반 점수화 및 상위 정책 선별
- **Knowledge Graph 파이프라인** — 문서 로드 → KG 생성 → Transform 적용
- **테스트셋 생성 파이프라인** — 페르소나 정의 → Single-hop 시나리오 → Instruction 적용 → 테스트셋 생성/저장
- **평가 파이프라인** — RAGAS 메트릭 기반 평가 및 결과 저장

## 설계 포인트
- **키워드·연령·위치 3중 점수화**: 단순 키워드 매칭이 아니라 청년 유형별 키워드, 연령 패턴, 제목/배지 내 키워드 위치까지 가중치로 합산해 청년 정책만 정교하게 필터링합니다.
- **커스텀 Single-hop Synthesizer**: `ragas`의 기본 Single-hop 로직을 상속해, 페르소나와 문서 키워드를 매칭시켜 단일 문서 기반 질문을 생성하도록 커스터마이징했습니다.
- **다중쿼리 노트북과의 차이**: 두 문서를 종합해야 하는 Multi-hop 질문 대신, 한 문서만으로 답변 가능한 Single-hop 질문을 생성하도록 Synthesizer와 instruction을 별도로 구성했습니다.
- **Threshold 기반 평가**: 단순 평균 점수만 보지 않고, 메트릭별 Acceptable Threshold를 설정해 항목별 합격/불합격을 판정합니다.


**1. 청년 정책 필터링 및 저장**


In [ ]:
import json
import re
from typing import List, Dict, Tuple
from dataclasses import dataclass
from datetime import datetime

# =============================================================================
# 1. 청년 유형별 키워드 정의
# =============================================================================

YOUTH_JOB_SEEKER = [
    "취업준비", "취업준비생", "미취업", "실업",
    "구직", "일자리", "취업",
    "이력서", "면접", "면접비",
    "취업교육", "직업훈련", "국비지원",
    "자격증", "시험응시료", "어학",
    "청년구직", "청년취업"
]

YOUTH_STUDENT = [
    "대학생", "대학원생", "재학생",
    "휴학", "복학", "생활비",
    "등록금", "장학금", "학자금",
    "기숙사", "교재비", "교통비", "멘토링"
]

YOUTH_EARLY_WORKER = [
    "사회초년생", "금융", "저소득",
    "재직청년", "근로청년",
    "중소기업", "중견기업",
    "저축", "근로환경", "직무능력", "상환"
]

YOUTH_INDEPENDENT = [
    "자취", "주택",
    "1인가구",
    "월세", "전세", "보증금",
    "공과금",
    "이사비", "주거비"
]

YOUTH_NEWLYWED = [
    "결혼", "신혼", "신혼부부",
    "예비부부", "무주택자",
    "혼인", "혼인신고",
    "주거마련", "행복주택",
    "전세자금", "주택자금", "출산", "양육지원금", "산후조리",
    "검진", "검사", "임산부", "산모"
]

YOUTH_STARTUP = [
    "창업", "청년창업", "예비창업",
    "1인창업", "자영업자",
    "고용보험료", "대출",
    "창업자금", "운영자금",
    "재창업", "폐업", "임대료"
]

YOUTH_FREELANCER = [
    "프리랜서", "비정규직",
    "계약직", "단기근로",
    "사업자"
]

YOUTH_LOW_INCOME = [
    "저소득", "차상위",
    "기초생활수급", "수급자",
    "경제적취약", "소득하위",
    "생활비", "긴급지원",
    "금융취약", "채무",
    "연체", "신용회복", "건강보험료"
]

YOUTH_MENTAL = [
    "고립", "은둔",
    "심리상담", "정신건강",
    "상담",
    "회복", "정서지원"
]

YOUTH_MILITARY = [
    "군복무", "복무",
    "전역", "제대군인",
    "복학"
]

YOUTH_RURAL = [
    "농어촌", "귀농", "귀촌",
    "청년농업인", "청년농",
    "농업", "축산", "농어업인",
    "농촌정착", "영농"
]

YOUTH_VULNERABLE = [
    "장애", "장애청년",
    "다문화", "외국인",
    "한부모",
    "차별", "언어",
    "통역", "한국어"
]

YOUTH_TYPE_KEYWORDS = {
    "취업준비청년": YOUTH_JOB_SEEKER,
    "대학생·대학원생": YOUTH_STUDENT,
    "사회초년생": YOUTH_EARLY_WORKER,
    "자취·독립청년": YOUTH_INDEPENDENT,
    "신혼청년": YOUTH_NEWLYWED,
    "창업청년": YOUTH_STARTUP,
    "프리랜서·플랫폼청년": YOUTH_FREELANCER,
    "저소득청년": YOUTH_LOW_INCOME,
    "심리취약청년": YOUTH_MENTAL,
    "군복무·전역청년": YOUTH_MILITARY,
    "농어촌청년": YOUTH_RURAL,
    "취약계층청년": YOUTH_VULNERABLE
}

# =============================================================================
# 2. 청년 정책 점수화 클래스
# =============================================================================

@dataclass
class PolicyScore:
    """정책 점수 데이터 클래스"""
    doc_id: str
    title: str
    score: float
    keyword_score: float
    age_score: float
    location_score: float
    youth_types: List[str]
    content: str
    metadata: dict

class YouthPolicyScorer:
    """청년 정책 점수화 시스템"""

    def __init__(self):
        # 청년 핵심 키워드 (가중치) - 기본 키워드
        self.core_youth_keywords = {
            # 매우 강한 청년 지표
            "청년": 10,
            "만 19세": 10, "만19세": 10, "만 34세": 10, "만34세": 10,
            "19세~34세": 10, "19~34세": 10, "19-34세": 10,

            # 연령 관련
            "만 18세": 8, "만18세": 8, "만 39세": 8, "만39세": 8,
        }

        # 유형별 키워드 가중치 설정
        self.youth_type_weights = {
            "취업준비청년": 7,
            "대학생·대학원생": 7,
            "사회초년생": 6,
            "자취·독립청년": 6,
            "신혼청년": 6,
            "창업청년": 7,
            "프리랜서·플랫폼청년": 5,
            "저소득청년": 6,
            "심리취약청년": 5,
            "군복무·전역청년": 5,
            "농어촌청년": 5,
            "취약계층청년": 6
        }

        # 제외 키워드 (강력)
        self.exclude_keywords = [
            "노인", "어르신", "경로", "노령", "고령자", "65세 이상",
            "아동", "영유아", "유치원", "어린이집", "미취학",
            "초등학생", "중학생", "고등학생", "미성년",
            "장애인복지", "중증장애", "발달장애", "마감"
        ]

        # 연령 패턴
        self.age_patterns = [
            r"만\s*(\d+)\s*세?\s*~\s*만?\s*(\d+)\s*세",
            r"만?\s*(\d+)\s*세?\s*이상\s*만?\s*(\d+)\s*세?\s*이하",
            r"(\d+)\s*세?\s*~\s*(\d+)\s*세",
            r"만\s*(\d+)\s*세",
            r"(\d+)\s*세\s*이상"
        ]

    def should_exclude(self, text: str) -> bool:
        """제외 키워드 체크"""
        text_lower = text.lower()
        for keyword in self.exclude_keywords:
            if keyword in text_lower:
                return True
        return False

    def extract_text_parts(self, content: str) -> Dict[str, str]:
        """문서에서 주요 부분 추출"""
        parts = {
            "title": "",
            "keywords": "",
            "target": "",
            "content_full": content[:1000]
        }

        # 제목 추출
        title_match = re.search(r"#\s*(.+?)\n", content)
        if title_match:
            parts["title"] = title_match.group(1).strip()

        # 배지/키워드 추출
        keyword_match = re.search(r"-\s*배지:\s*(.+?)\n", content)
        if keyword_match:
            parts["keywords"] = keyword_match.group(1).strip()

        # 지원대상 추출
        target_match = re.search(r"\[지원대상\]\s*(.+?)(?:\n\[|$)", content, re.DOTALL)
        if target_match:
            parts["target"] = target_match.group(1).strip()[:500]

        return parts

    def detect_youth_types(self, text: str) -> List[str]:
        """텍스트에서 청년 유형 감지"""
        detected_types = []
        text_lower = text.lower()

        for youth_type, keywords in YOUTH_TYPE_KEYWORDS.items():
            for keyword in keywords:
                if keyword.lower() in text_lower:
                    detected_types.append(youth_type)
                    break

        return list(set(detected_types))

    def calculate_keyword_score(self, text: str, youth_types: List[str]) -> float:
        """키워드 기반 점수 계산 (핵심 + 유형별)"""
        score = 0.0
        text_lower = text.lower()

        # 1) 핵심 청년 키워드 점수
        for keyword, weight in self.core_youth_keywords.items():
            count = text_lower.count(keyword.lower())
            if count > 0:
                score += weight * min(count, 3)

        # 2) 유형별 키워드 점수
        for youth_type in youth_types:
            base_weight = self.youth_type_weights.get(youth_type, 5)
            keywords = YOUTH_TYPE_KEYWORDS.get(youth_type, [])

            for keyword in keywords:
                count = text_lower.count(keyword.lower())
                if count > 0:
                    # 유형별 가중치 적용
                    score += base_weight * min(count, 2)

        return score

    def calculate_age_score(self, text: str) -> float:
        """연령 기반 점수 계산"""
        score = 0.0

        for pattern in self.age_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                if isinstance(match, tuple):
                    ages = [int(age) for age in match if age.isdigit()]
                else:
                    ages = [int(match)] if match.isdigit() else []

                for age in ages:
                    if 19 <= age <= 34:
                        score += 20
                    elif 18 <= age <= 39:
                        score += 10
                    elif age < 19:
                        score -= 10
                    elif age >= 60:
                        score -= 20

        return score

    def calculate_location_score(self, parts: Dict[str, str], youth_types: List[str]) -> float:
        """키워드 위치 기반 점수 계산"""
        score = 0.0

        # 제목에 청년 키워드 (가중치 3배)
        title_keywords = ["청년", "대학생", "신혼", "취업", "창업"]
        for keyword in title_keywords:
            if keyword in parts["title"]:
                score += 30

        # 배지/키워드에 청년
        if "청년" in parts["keywords"]:
            score += 20

        # 지원대상에 청년 키워드
        if "청년" in parts["target"] or "대학생" in parts["target"]:
            score += 15

        # 유형이 많을수록 가산점
        score += len(youth_types) * 5

        return score

    def score_policy(self, item: dict) -> PolicyScore:
        """개별 정책 점수 계산"""
        content = item.get("content", "")
        parts = self.extract_text_parts(content)

        # 제외 키워드 체크
        if self.should_exclude(parts["title"] + parts["keywords"] + parts["target"]):
            return PolicyScore(
                doc_id=item.get("doc_id", ""),
                title=parts["title"],
                score=0.0,
                keyword_score=0.0,
                age_score=0.0,
                location_score=0.0,
                youth_types=[],
                content=content,
                metadata=item.get("metadata", {})
            )

        # 청년 유형 감지
        youth_types = self.detect_youth_types(
            parts["title"] + " " + parts["keywords"] + " " + parts["target"] + " " + content[:500]
        )

        # 각 점수 계산
        keyword_score = self.calculate_keyword_score(
            parts["title"] + " " + parts["keywords"] + " " + parts["target"],
            youth_types
        )
        age_score = self.calculate_age_score(content)
        location_score = self.calculate_location_score(parts, youth_types)

        # 최종 점수
        total_score = (
            keyword_score * 1.0 +
            age_score * 0.5 +
            location_score * 1.5
        )

        return PolicyScore(
            doc_id=item.get("doc_id", ""),
            title=parts["title"],
            score=total_score,
            keyword_score=keyword_score,
            age_score=age_score,
            location_score=location_score,
            youth_types=youth_types,
            content=content,
            metadata=item.get("metadata", {})
        )

# =============================================================================
# 3. 청년 정책 필터링 함수
# =============================================================================

def filter_youth_policies(
    data: List[dict],
    target_count: int = 10,
    score_threshold: float = 10.0
) -> Tuple[List[dict], List[PolicyScore]]:
    """청년 관련 정책 필터링"""
    print(f"\n{'='*80}")
    print(f"🔍 청년 정책 필터링 시작")
    print(f"{'='*80}")

    scorer = YouthPolicyScorer()

    # 1단계: 전체 문서 점수 계산
    print(f"\n📊 1단계: 전체 문서 점수 계산 중...")
    scored_policies = []

    for i, item in enumerate(data):
        if (i + 1) % 500 == 0:
            print(f"   진행: {i+1}/{len(data)} 문서 처리 완료...")

        score_obj = scorer.score_policy(item)

        if score_obj.score >= score_threshold:
            scored_policies.append(score_obj)

    print(f"✅ 점수 계산 완료: {len(scored_policies)}개 정책이 기준 충족")

    # 2단계: 점수 순 정렬
    print(f"\n📈 2단계: 점수 순 정렬 중...")
    scored_policies.sort(key=lambda x: x.score, reverse=True)

    # 3단계: 상위 N개 선택
    print(f"\n✂️ 3단계: 상위 {target_count}개 선택 중...")
    top_policies = scored_policies[:target_count]

    # 통계 출력
    print(f"\n{'='*80}")
    print(f"📊 필터링 결과 통계")
    print(f"{'='*80}")
    print(f"전체 문서 수: {len(data):,}개")
    print(f"기준 충족 문서: {len(scored_policies):,}개")
    print(f"최종 선택: {len(top_policies):,}개")

    if top_policies:
        print(f"\n점수 범위:")
        print(f"   최고 점수: {top_policies[0].score:.1f}")
        print(f"   최저 점수: {top_policies[-1].score:.1f}")
        print(f"   평균 점수: {sum(p.score for p in top_policies) / len(top_policies):.1f}")

    # 원본 데이터 형태로 변환
    filtered_data = []
    for score_obj in top_policies:
        for item in data:
            if item.get("doc_id") == score_obj.doc_id:
                filtered_data.append(item)
                break

    return filtered_data, top_policies

# =============================================================================
# 4. 실행 코드
# =============================================================================

if __name__ == "__main__":
    print("📂 복지 정책 데이터 로드 중...")

    input_file = "/content/타겟사이트트_서비스 목록_민간.json"

    try:
        with open(input_file, "r", encoding="utf-8") as f:
            all_data = json.load(f)

        print(f"✅ 전체 정책 로드 완료: {len(all_data):,}개")

        # 청년 정책 필터링
        youth_data, score_objects = filter_youth_policies(
            data=all_data,
            target_count=10,
            score_threshold=10.0
        )

        # 결과 확인
        print(f"\n{'='*80}")
        print(f"📋 상위 10개 청년 정책 미리보기")
        print(f"{'='*80}\n")

        for i, score_obj in enumerate(score_objects[:10], 1):
            print(f"{i:2d}. [{score_obj.score:6.1f}점] {score_obj.title}")
            print(f"    키워드: {score_obj.keyword_score:.1f} | 연령: {score_obj.age_score:.1f} | 위치: {score_obj.location_score:.1f}")
            print(f"    청년유형: {', '.join(score_obj.youth_types) if score_obj.youth_types else '없음'}")
            print()

        # 결과 저장
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        # 1) 필터링된 정책 데이터 저장
        output_file = 'youth_policies_filtered.json'
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(youth_data, f, ensure_ascii=False, indent=2)
        print(f"\n✅ 청년 정책 데이터 저장: {output_file}")

        # 2) 점수 정보 저장
        scores_file = 'youth_policies_scores.json'
        scores_data = [
            {
                "doc_id": s.doc_id,
                "title": s.title,
                "total_score": round(s.score, 2),
                "keyword_score": round(s.keyword_score, 2),
                "age_score": round(s.age_score, 2),
                "location_score": round(s.location_score, 2),
                "youth_types": s.youth_types
            }
            for s in score_objects
        ]

        with open(scores_file, 'w', encoding='utf-8') as f:
            json.dump(scores_data, f, ensure_ascii=False, indent=2)
        print(f"✅ 점수 정보 저장: {scores_file}")

        print(f"\n{'='*80}")
        print("✅ 모든 작업 완료!")
        print(f"{'='*80}")

    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {input_file}")
    except Exception as e:
        print(f"❌ 오류 발생: {str(e)}")

📂 복지 정책 데이터 로드 중...
✅ 전체 정책 로드 완료: 93개

🔍 청년 정책 필터링 시작

📊 1단계: 전체 문서 점수 계산 중...
✅ 점수 계산 완료: 48개 정책이 기준 충족

📈 2단계: 점수 순 정렬 중...

✂️ 3단계: 상위 10개 선택 중...

📊 필터링 결과 통계
전체 문서 수: 93개
기준 충족 문서: 48개
최종 선택: 10개

점수 범위:
   최고 점수: 237.5
   최저 점수: 110.5
   평균 점수: 149.2

📋 상위 10개 청년 정책 미리보기

 1. [ 237.5점] 취업역량강화
    키워드: 95.0 | 연령: 60.0 | 위치: 75.0
    청년유형: 취업준비청년, 대학생·대학원생

 2. [ 207.5점] 마이크로크레딧 사업
    키워드: 105.0 | 연령: 40.0 | 위치: 55.0
    청년유형: 프리랜서·플랫폼청년, 심리취약청년, 취업준비청년, 사회초년생, 저소득청년, 취약계층청년, 창업청년

 3. [ 163.0점] 대학생 경제·금융 워크숍 및 프리젠테이션 대회
    키워드: 43.0 | 연령: 0.0 | 위치: 80.0
    청년유형: 취업준비청년, 대학생·대학원생, 사회초년생

 4. [ 139.5점] 다문화가족 인재DB
    키워드: 67.0 | 연령: 40.0 | 위치: 35.0
    청년유형: 취업준비청년, 취약계층청년, 신혼청년

 5. [ 135.5점] 창업지원
    키워드: 38.0 | 연령: 0.0 | 위치: 65.0
    청년유형: 심리취약청년, 취업준비청년, 창업청년

 6. [ 132.5점] 영농정착
    키워드: 65.0 | 연령: 0.0 | 위치: 45.0
    청년유형: 취업준비청년, 농어촌청년, 사회초년생, 자취·독립청년, 창업청년

 7. [ 128.5점] HUG프로그램
    키워드: 31.0 | 연령: 60.0 | 위치: 45.0
    청년유형: 취업준비청년, 대학생·대학원생

 8. [ 121.0점] 대학생아시아대장정
    키워드

**2. 필요 라이브러리 설치**


In [ ]:
!pip install ragas
!pip install langchain
!pip install langchain_community
!pip install langchain-openai
!pip install jq

**3. LangChain Document 변환**


In [55]:
from langchain_core.documents import Document
import json

# JSON 파일 읽기 (자동 감지)
documents = []
# youth_policies_filtered.json
with open("youth_policies_filtered.json", "r", encoding="utf-8") as f:
    try:
        # 먼저 전체를 JSON으로 파싱 시도
        data = json.load(f)

        # 딕셔너리면 리스트로 변환
        if isinstance(data, dict):
            data = [data]

        # 리스트 처리
        for item in data:
            documents.append(
                Document(
                    page_content=item["content"],
                    metadata={
                        "doc_id": item.get("doc_id", ""),
                        "created_at": item.get("created_at", ""),
                        "source_name": item.get("source_name", ""),
                        "source_type": item.get("source_type", ""),
                        "source_file": item.get("source_file", "")
                    }
                )
            )

    except json.JSONDecodeError:
        # JSON 파싱 실패 시 JSONL로 시도
        f.seek(0)  # 파일 포인터 처음으로
        for line in f:
            if line.strip():
                item = json.loads(line)
                documents.append(
                    Document(
                        page_content=item["content"],
                        metadata={
                            "doc_id": item.get("doc_id", ""),
                            "created_at": item.get("created_at", ""),
                            "source_name": item.get("source_name", ""),
                            "source_type": item.get("source_type", ""),
                            "source_file": item.get("source_file", "")
                        }
                    )
                )

print(f"총 문서 수: {len(documents)}")
if documents:
    print("\n첫 번째 문서 미리보기:")
    print(f"Doc ID: {documents[0].metadata['doc_id']}")
    print(f"Content: {documents[0].page_content[:200]}...")
else:
    print("⚠️ 문서를 찾을 수 없습니다. 파일 형식을 확인해주세요.")

총 문서 수: 10

첫 번째 문서 미리보기:
Doc ID: 복지서비스_서비스 목록_민간_0031
Content: # 취업역량강화

- 배지: 민간 복지서비스, 일자리, 청년
- 담당부처: 남북하나재단
- 사업상태: 진행중
- 사업기간: 2014-01-01 ~ 2099-12-31
- 연락처: 0215776635

[사업목적]
null

[지원대상]
- 청년취업아카데미: 대학생 중심 청년 구직(예정)자 대상 취업교육알선 프로그램 제공 
- 청년세대 취업지원 바우처: 만...


**4. LLM·임베딩 초기화**


In [58]:
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI
from ragas.embeddings import OpenAIEmbeddings
import openai

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o",openai_api_key= api_key))
# 올바른 OpenAI Client 생성
critic_client = openai.OpenAI(api_key= api_key)

# Embeddings 연결
generator_embeddings = OpenAIEmbeddings(client=critic_client)


/tmp/ipython-input-1884700722.py:6: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o",openai_api_key= api_key))


**5. 유형별 페르소나 정의**


In [ ]:
#각 유형별 페르소나 정리 

from ragas.testset.persona import Persona

# 1. 취업준비생 유형
person1 = Persona(
    name="jobseeker_final_year",
    role_description="4학년 취업준비생으로 학자금 대출 상환 시기가 다가오고 있으며, 면접 정장 구입비와 자격증 취득 비용 때문에 고민이 많습니다. 청년 취업 지원금과 학자금 상환 유예 제도에 관심이 많습니다.",
)

# 2. 사회초년생 유형
person2 = Persona(
    name="first_job_low_income",
    role_description="중소기업에 입사한 지 3개월 된 사회초년생으로 월급 220만원을 받습니다. 자취를 시작하려는데 보증금 마련이 어려워 청년 전월세 대출과 월세 지원에 관심이 많습니다.",
)

# 3. 자취·독립 청년 유형
person3 = Persona(
    name="single_household_goshiwon",
    role_description="고시원에서 자취하는 1인 가구 청년으로 월세 40만원이 부담됩니다. 더 나은 주거환경으로 이사하고 싶지만 보증금이 없어 청년 주거 지원 정책을 알아보고 있습니다.",
)

# 4. 대학생/대학원생 유형
person4 = Persona(
    name="undergraduate_part_timer",
    role_description="등록금과 생활비를 스스로 벌어야 하는 대학생으로 편의점 아르바이트를 하고 있습니다. 학업과 아르바이트 병행이 힘들어 장학금이나 근로장학금을 알아보고 있습니다.",
)

# 5. 군필·전역 예정 청년 유형
person5 = Persona(
    name="recently_discharged",
    role_description="전역한 지 2개월 된 청년으로 사회 복귀 준비 중입니다. 취업 준비 비용이 필요해 전역 장병 지원금이나 취업 지원 프로그램을 알아보고 있습니다.",
)

# 6. 프리랜서·플랫폼 노동 청년 유형
person6 = Persona(
    name="freelance_designer",
    role_description="프리랜서 디자이너로 일하며 소득이 불규칙합니다. 4대 보험 가입이 안 되어 있고 건강보험료 부담이 커서 프리랜서 지원 제도를 찾고 있습니다.",
)

# 7. 1인 창업·예비창업 청년 유형
person7 = Persona(
    name="cafe_startup_founder",
    role_description="소규모 카페를 창업한 지 6개월 된 청년으로 초기 자금 압박이 큽니다. 청년 창업 자금 지원이나 점포 임대료 지원에 관심이 많습니다.",
)

# 8. 저소득·취약·금융취약 청년 유형
person8 = Persona(
    name="low_income_worker",
    role_description="월 180만원을 버는 저소득 청년으로 생계유지가 어렵습니다. 긴급 생활비 지원이나 청년 생활안정 지원금을 받을 수 있는지 알아보고 있습니다.",
)

# 9. 장애 청년 유형
person9 = Persona(
    name="mild_disability_jobseeker",
    role_description="경증 장애가 있는 취업준비생으로 장애인 고용 지원 제도에 관심이 많습니다. 장애인 취업 지원금이나 근로 지원 서비스를 알아보고 있습니다.",
)

# 10. 외국인·다문화 청년 유형
person10 = Persona(
    name="multicultural_korean",
    role_description="다문화 가정 출신 청년으로 한국어는 능숙하지만 취업 시 차별을 경험했습니다. 다문화 청년 지원 제도나 취업 지원 프로그램을 알고 싶어합니다.",
)

# 11. 경력전환·재취업 청년 유형
person11 = Persona(
    name="career_changer",
    role_description="직장을 그만두고 새로운 분야로 전환을 준비하는 청년입니다. 직업훈련 지원이나 경력전환 교육비 지원 제도를 알아보고 있습니다.",
)

# 12. 결혼·임신·가족계획 청년 유형
person12 = Persona(
    name="newlywed_couple",
    role_description="결혼 1년차 신혼부부로 전세자금 마련에 어려움을 겪고 있습니다. 신혼부부 주거 지원이나 전세자금 대출 우대를 알아보고 있습니다.",
)

# 13. 농어촌 청년 유형
person13 = Persona(
    name="rural_farmer",
    role_description="귀농한 청년 농부로 초기 정착 자금과 농업 기술 습득이 필요합니다. 청년 농업인 지원금이나 영농 정착 지원 제도를 찾고 있습니다.",
)

# 리스트로 모음
persona_list = [
    person1, person2, person3, person4, person5, person6, person7,
    person8, person9, person10, person11, person12, person13
]


**6. Knowledge Graph 생성**


In [60]:
from ragas.testset.graph import KnowledgeGraph
from ragas.testset.graph import Node, NodeType
# KnowledgeGraph에 추가
kg = KnowledgeGraph()
for doc in documents:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={
                "page_content": doc.page_content,
                "document_metadata": doc.metadata,
            },
        )
    )


**7. Knowledge Graph 노드 확인**


In [61]:
print(len(kg.nodes))
print(kg.nodes[0].properties.keys())

10
dict_keys(['page_content', 'document_metadata'])


**8. 라이브러리 설치 (rapidfuzz)**


In [ ]:
!pip install rapidfuzz

**9. 문서 Transform 적용**


In [65]:
from ragas.testset.transforms import Parallel, apply_transforms
from ragas.testset.transforms import (
    HeadlinesExtractor,
    KeyphrasesExtractor,
    OverlapScoreBuilder,
)


headline_extractor = HeadlinesExtractor(llm=generator_llm)
keyphrase_extractor = KeyphrasesExtractor(
    llm=generator_llm, property_name="keyphrases", max_num=10
)
relation_builder = OverlapScoreBuilder(
    property_name="keyphrases",
    new_property_name="overlap_score",
    threshold=0.01,
    distance_threshold=0.9,
)

transforms = [
    headline_extractor,
    keyphrase_extractor,
    relation_builder,
]

apply_transforms(kg, transforms=transforms)

# Diagnostic: Check if keyphrases_overlap relationships exist
overlap_relationships = [rel for rel in kg.relationships if rel.type == "keyphrases_overlap"]
print(f"생성된 'keyphrases_overlap' 관계 수: {len(overlap_relationships)}")
if overlap_relationships:
    print(f"첫 번째 'keyphrases_overlap' 관계 예시: {overlap_relationships[0].properties}")

Applying HeadlinesExtractor:   0%|          | 0/10 [00:00<?, ?it/s]

Applying KeyphrasesExtractor:   0%|          | 0/10 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

생성된 'keyphrases_overlap' 관계 수: 3
첫 번째 'keyphrases_overlap' 관계 예시: {'keyphrases_overlap_score': 0.023809523809523808, 'overlapped_items': [('일자리', '일자리')]}


**10. Single-hop 시나리오 생성**


In [66]:
from ragas.testset.synthesizers.single_hop import (
    SingleHopQuerySynthesizer,
    SingleHopScenario,
)
from dataclasses import dataclass
from ragas.testset.synthesizers.prompts import (
    ThemesPersonasInput,
    ThemesPersonasMatchingPrompt,
)


@dataclass
class MySingleHopScenario(SingleHopQuerySynthesizer):

    theme_persona_matching_prompt = ThemesPersonasMatchingPrompt()

    async def _generate_scenarios(self, n, knowledge_graph, persona_list, callbacks):

        property_name = "headlines"
        nodes = []
        for node in knowledge_graph.nodes:
            if node.get_property(property_name):
                nodes.append(node)

        number_of_samples_per_node = max(1, n // len(nodes))

        scenarios = []
        for node in nodes:
            if len(scenarios) >= n:
                break
            themes = node.properties.get(property_name, [""])
            prompt_input = ThemesPersonasInput(themes=themes, personas=persona_list)
            persona_concepts = await self.theme_persona_matching_prompt.generate(
                data=prompt_input, llm=self.llm, callbacks=callbacks
            )
            base_scenarios = self.prepare_combinations(
                node,
                themes,
                personas=persona_list,
                persona_concepts=persona_concepts.mapping,
            )
            scenarios.extend(
                self.sample_combinations(base_scenarios, number_of_samples_per_node)
            )

        return scenarios

query = MySingleHopScenario(llm=generator_llm)

scenarios = await query.generate_scenarios(
    n=5, knowledge_graph=kg, persona_list=persona_list
)

scenarios[0]

SingleHopScenario(
nodes=1
term=[지원내용]
persona=name='career_changer' role_description='직장을 그만두고 새로운 분야로 전환을 준비하는 청년입니다. 직업훈련 지원이나 경력전환 교육비 지원 제도를 알아보고 있습니다.'
style=QueryStyle.WEB_SEARCH_LIKE
length=QueryLength.MEDIUM)

**11. 시나리오 확인**


In [67]:
scenarios[::]

[SingleHopScenario(
 nodes=1
 term=[지원내용]
 persona=name='career_changer' role_description='직장을 그만두고 새로운 분야로 전환을 준비하는 청년입니다. 직업훈련 지원이나 경력전환 교육비 지원 제도를 알아보고 있습니다.'
 style=QueryStyle.WEB_SEARCH_LIKE
 length=QueryLength.MEDIUM),
 SingleHopScenario(
 nodes=1
 term=[지원대상]
 persona=name='multicultural_korean' role_description='다문화 가정 출신 청년으로 한국어는 능숙하지만 취업 시 차별을 경험했습니다. 다문화 청년 지원 제도나 취업 지원 프로그램을 알고 싶어합니다.'
 style=QueryStyle.POOR_GRAMMAR
 length=QueryLength.LONG),
 SingleHopScenario(
 nodes=1
 term=지원내용
 persona=name='career_changer' role_description='직장을 그만두고 새로운 분야로 전환을 준비하는 청년입니다. 직업훈련 지원이나 경력전환 교육비 지원 제도를 알아보고 있습니다.'
 style=QueryStyle.POOR_GRAMMAR
 length=QueryLength.MEDIUM),
 SingleHopScenario(
 nodes=1
 term=[지원대상]
 persona=name='rural_farmer' role_description='귀농한 청년 농부로 초기 정착 자금과 농업 기술 습득이 필요합니다. 청년 농업인 지원금이나 영농 정착 지원 제도를 찾고 있습니다.'
 style=QueryStyle.MISSPELLED
 length=QueryLength.LONG),
 SingleHopScenario(
 nodes=1
 term=[사업목적]
 persona=name='multicultural_korean' role_description='다문화 

**12. 추가 Transform 정의**


In [73]:
from ragas.testset.transforms.extractors.llm_based import NERExtractor
from ragas.testset.transforms.splitters import HeadlineSplitter

transforms = [HeadlineSplitter(), NERExtractor(llm = generator_llm )]

**13. Testset Generator 초기화**


In [74]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(knowledge_graph=kg, persona_list=persona_list, llm=generator_llm, embedding_model = generator_embeddings)

**14. 질문/답변 생성 Instruction 정의**


In [76]:
instruction = """
당신은 대한민국 청년 복지 정책에 정통한 공공 콜봇 전문가입니다.
제공된 문서(context)와 페르소나(persona)를 참고하여 실제 사용자가 묻는 것처럼 자연스러운 질문과 답변을 생성하세요.

### 질문 생성 규칙
1. 페르소나의 상황과 고민을 반영한 구체적인 질문을 만드세요
   - 예: "저는 [페르소나 상황]인데, [구체적 고민]에 대해 궁금합니다"
2. 정책명이나 제도명은 직접 언급하지 말고, 상황 설명으로 대체하세요
3. 질문은 단일 문서의 내용으로 답변 가능한 수준이어야 합니다 (Single-hop)
4. 실제 사용자가 복지 콜센터에 전화해서 물어보는 것처럼 자연스럽게 작성하세요

### 답변 생성 규칙
1. 답변은 **반드시 제공된 문서의 내용만** 사용하세요
2. 문서에 없는 조건, 해석, 추론은 절대 추가하지 마세요
3. 수치(연령, 금액, 소득 기준 등)는 문서에 명시된 것만 정확히 포함하세요
4. 공공기관 상담사가 안내하는 것처럼 명확하고 친절하게 작성하세요
5. 페르소나의 상황을 먼저 요약한 후 답변하세요
   - 예: "귀하의 경우 [상황 요약]이시군요. [답변 내용]"
6. 사용자가 이용할수 있는 정책명, 기관 등을 언급하세요.
7. 지자체 복지 서비스는 해당 지자체에 주소지를 둔 주민만 수혜 가능하다는 전제를 따른다. 따라서 사용자의 현재 거주 지역(주소지) 외 타 지역에서 제공하는 복지 서비스의 수혜 가능 여부를 묻는 질문은 생성하지 않는다. 질문에 등장하는 모든 지역 기반 복지 서비스는 사용자의 거주 지역과 일치해야 한다.


### 예시
"질문": "저는 현재 실업급여를 받고 있으며, 구직 과정에서 자신감이 많이 떨어져 있고 면접이나 직무 선택에도 어려움을 느끼고 있습니다. 이런 상황에서 심리 상담이나 집단 상담, 취업 관련 특강 같은 지원을 받을 수 있는지 궁금합니다.동시에 직업훈련에 참여하면서 훈련에 전념할 수 있도록 매달 일정 금액의 수당도 함께 받을 수 있는지 알고 싶은데, 실업급여를 받고 있는 상태에서도 이런 금전적 지원이 가능한가요?.",
"답변": "실업급여 수급자는 취약계층취업촉진 사업에 참여하여 전문심리상담, 집단상담, 취업특강 등 구직자 취업역량강화 프로그램을 통해 자신감 회복 및 구직기술 향상을 지원받을 수 있습니다. 그러나 직업능력개발훈련에서는 실업급여 수급자는 지원받지 못하며, 다른 법령에 의해 훈련수당이나 훈련장려금을 지원받고 있는 경우에도 지원이 제한됩니다.",
"""

**15. Instruction 프롬프트 적용**


In [77]:
from ragas.testset.synthesizers.single_hop.specific import (
    SingleHopSpecificQuerySynthesizer,
)
distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm, property_name='headlines'), 1.0),
]

for query_synthesizer, _ in distribution:
    prompts = await query_synthesizer.adapt_prompts("korean", llm=generator_llm)
    if "query_answer_generation_prompt" in prompts:
        prompts["query_answer_generation_prompt"].instruction = instruction
    query_synthesizer.set_prompts(**prompts)

print("✅ 커스텀 instruction 반영 완료")

✅ 커스텀 instruction 반영 완료


**16. 테스트셋 생성 실행**


In [82]:
# Initialize the Testset Generator
testset_generator = TestsetGenerator(knowledge_graph=kg, persona_list=persona_list, llm=generator_llm, embedding_model = generator_embeddings)
# Generate the Testset
print("테스트셋 생성 중... 잠시만 기다려 주세요.")
testset = testset_generator.generate(testset_size=10, query_distribution=distribution) # 'await' 키워드 제거
print("테스트셋 생성 완료.")
testset

테스트셋 생성 중... 잠시만 기다려 주세요.


Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

테스트셋 생성 완료.


Testset(samples=[TestsetSample(eval_sample=SingleTurnSample(user_input='저는 다문화 가정 출신 청년인데, 취업 준비를 하면서 차별을 느꼈습니다. 청년 취업 지원 프로그램이나 바우처 같은 지원을 받을 수 있는지 궁금합니다.', retrieved_contexts=None, reference_contexts=['# 취업역량강화\n\n- 배지: 민간 복지서비스, 일자리, 청년\n- 담당부처: 남북하나재단\n- 사업상태: 진행중\n- 사업기간: 2014-01-01 ~ 2099-12-31\n- 연락처: 0215776635\n\n[사업목적]\nnull\n\n[지원대상]\n- 청년취업아카데미: 대학생 중심 청년 구직(예정)자 대상 취업교육알선 프로그램 제공 \r\n- 청년세대 취업지원 바우처: 만 18세 ~ 만 35세 이하 청년 취업 준비자 \r\n- 전문직종 특화사업: 만 18세 ~ 만 45세 이하 취업을 희망하는 북한이탈주민 \r\n- 온라인 배움터 자격취득 교육: 취업을 준비하는 북한이탈주민\n\n[지원내용]\n- 청년취업아카데미: 졸업자졸업예정자 중심 취업반과 재학생 중심 인턴십반 병행 운영 \r\n- 청년세대 취업지원 바우처: 취업에 필요한 학습지원을 위한 바우처 카드 발급 \r\n- 전문직종 특화사업: 기업의 현장수요 직종 선택으로 맞춤형 집중 훈련과정 개설 \r\n- 온라인 배움터 자격취득 교육: 북한이탈주민 교육생 전용 홈페이지를 통해 국가자격취득 및 정보화기술 등 300개 교육과정을 개설하여 온라인 교육으로 진행\n\n[신청방법]\n- 우편접수: 서울특별시 영등포구 은행로 54, 4층 (여의도동 12-3) \r\n- 연중 수시 접수\n\n[복지사례]\n\n'], retrieved_context_ids=None, reference_context_ids=None, response=None, multi_responses=None, reference='귀하의 경우 다문화 가정 출신 청년으로 취업 준비 중이시군

**17. 테스트셋 개수 확인**


In [84]:
len(testset)

10

**18. 테스트셋 결과 확인**


In [85]:
import textwrap

# 모든 샘플의 질문, 정답, 근거를 보고 싶을 때
if testset and testset.samples:
    for i, sample in enumerate(testset.samples):
        eval_sample = sample.eval_sample
        print(f"--- 샘플 {i+1} ---")
        print("질문:", eval_sample.user_input)

        # 정답 출력 (텍스트 래핑 적용)
        wrapped_reference = textwrap.fill(eval_sample.reference, width=100, initial_indent='  ', subsequent_indent='  ')
        print("정답:\n", wrapped_reference)

        # 근거 출력 (각 근거별로 텍스트 래핑 적용)
        print("근거:")
        if eval_sample.reference_contexts:
            for j, context in enumerate(eval_sample.reference_contexts):
                wrapped_context = textwrap.fill(context, width=100, initial_indent=f'  Context {j+1}: ', subsequent_indent='            ')
                print(wrapped_context)
        else:
            print("  (근거 없음)")

        print("\n") # 각 샘플 사이에 공백 추가
else:
    print("생성된 테스트 샘플이 없습니다.")

--- 샘플 1 ---
질문: 저는 다문화 가정 출신 청년인데, 취업 준비를 하면서 차별을 느꼈습니다. 청년 취업 지원 프로그램이나 바우처 같은 지원을 받을 수 있는지 궁금합니다.
정답:
   귀하의 경우 다문화 가정 출신 청년으로 취업 준비 중이시군요. 청년세대 취업지원 바우처를 통해 만 18세에서 만 35세 이하 청년 취업 준비자에게 취업에 필요한 학습지원을 위한
  바우처 카드를 발급받을 수 있습니다. 자세한 사항은 남북하나재단에 문의하시기 바랍니다.
근거:
  Context 1: # 취업역량강화  - 배지: 민간 복지서비스, 일자리, 청년 - 담당부처: 남북하나재단 - 사업상태: 진행중 - 사업기간: 2014-01-01 ~
            2099-12-31 - 연락처: 0215776635  [사업목적] null  [지원대상] - 청년취업아카데미: 대학생 중심 청년 구직(예정)자 대상
            취업교육알선 프로그램 제공   - 청년세대 취업지원 바우처: 만 18세 ~ 만 35세 이하 청년 취업 준비자   - 전문직종 특화사업: 만 18세 ~ 만
            45세 이하 취업을 희망하는 북한이탈주민   - 온라인 배움터 자격취득 교육: 취업을 준비하는 북한이탈주민  [지원내용] - 청년취업아카데미: 졸업자졸업예정자
            중심 취업반과 재학생 중심 인턴십반 병행 운영   - 청년세대 취업지원 바우처: 취업에 필요한 학습지원을 위한 바우처 카드 발급   - 전문직종 특화사업:
            기업의 현장수요 직종 선택으로 맞춤형 집중 훈련과정 개설   - 온라인 배움터 자격취득 교육: 북한이탈주민 교육생 전용 홈페이지를 통해 국가자격취득 및
            정보화기술 등 300개 교육과정을 개설하여 온라인 교육으로 진행  [신청방법] - 우편접수: 서울특별시 영등포구 은행로 54, 4층 (여의도동 12-3)
            - 연중 수시 접수  [복지사례]


--- 샘플 2 ---
질문: 저는 다문

**19. 테스트셋 JSON 저장**


In [86]:
import json

# testset을 리스트로 변환
data = []
for sample in testset.samples:
    eval_sample = sample.eval_sample
    data.append({
        "질문": eval_sample.user_input,
        "정답": eval_sample.reference,
        "근거": eval_sample.reference_contexts
    })

# JSON 파일로 저장
with open("testset.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)


**20. 평가세트(RAGAS) 평가 실행**


(Colab에서 실행 / 결과는 CSV로 별도 다운로드됨)


In [ ]:
import json
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall
)
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # LangChain 것 사용!
from datasets import Dataset
import pandas as pd
import os

# -------------------
# LLM 및 Embeddings 초기화 (수정!)
# -------------------

# Critic용 LLM
critic_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o-mini", openai_api_key=api_key)
)

# Embeddings - LangChain의 OpenAIEmbeddings 사용!
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=api_key
)

# -------------------
# Metrics 설정
# -------------------
metrics = [
    Faithfulness(llm=critic_llm),
    AnswerRelevancy(llm=critic_llm, embeddings=embeddings),
    ContextPrecision(llm=critic_llm),
    ContextRecall(llm=critic_llm)
]

print("✅ Metrics 초기화 완료!")

# -------------------
# JSON 문서 읽기
# -------------------
with open("/content/testset.json", "r", encoding="utf-8") as f:
    testset_korean = json.load(f)

print(f"✅ Testset 로드 완료: {len(testset_korean)}개")

# 한글 키를 영어 키로 변환
eval_data = {
    'question': [],
    'answer': [],
    'contexts': [],
    'ground_truth': []
}

print("🔄 데이터 변환 중...")

for item in testset_korean:
    question = item.get('질문', '')
    answer = item.get('정답', '')
    contexts = item.get('근거', [])
    ground_truth = item.get('정답', '')

    if isinstance(contexts, str):
        contexts = [contexts]

    if not question or not answer:
        print(f"⚠️ 경고: 빈 질문 또는 답변 발견, 건너뜀")
        continue

    eval_data['question'].append(question)
    eval_data['answer'].append(answer)
    eval_data['contexts'].append(contexts)
    eval_data['ground_truth'].append(ground_truth)

print(f"✅ 변환 완료: {len(eval_data['question'])}개 항목")

# Dataset 생성
eval_dataset = Dataset.from_dict(eval_data)

print("\n📋 데이터 미리보기:")
print(f"   질문 1: {eval_data['question'][0][:80]}...")
print(f"   답변 1: {eval_data['answer'][0][:80]}...")
print(f"   근거 개수: {len(eval_data['contexts'][0])}")

# -------------------
# 평가 실행
# -------------------
print("\n🔄 RAGAS 평가 실행 중...")
print(f"   총 {len(eval_dataset)}개 항목 평가")
print("   ☕ 시간이 소요될 수 있습니다 (예상: 5-10분)...\n")

results = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=critic_llm,
    embeddings=embeddings
)

print("\n✅ 평가 완료!")

# -------------------
# 결과 확인 (수정!)
# -------------------
print("\n" + "="*80)
print("📊 RAGAS 평가 결과")
print("="*80)

# DataFrame으로 변환
results_df = results.to_pandas()

# DataFrame 컬럼 확인
print("\n📋 결과 DataFrame 컬럼:")
print(results_df.columns.tolist())
print()

# 평균 점수 출력
print("📈 메트릭별 평균 점수:\n")
for metric in ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']:
    if metric in results_df.columns:
        score = results_df[metric].mean()
        if pd.notna(score):  # NaN 체크
            emoji = "✅" if score >= 0.8 else "⚠️" if score >= 0.7 else "❌"
            print(f"   {emoji} {metric.replace('_', ' ').title()}: {score:.4f}")
        else:
            print(f"   ❌ {metric.replace('_', ' ').title()}: 평가 실패 (NaN)")
    else:
        print(f"   ⚠️ {metric.replace('_', ' ').title()}: 컬럼 없음")

# 원본 데이터와 결과 병합
print("\n" + "="*80)
print("📋 개별 평가 결과 (처음 5개)")
print("="*80)

for idx in range(min(5, len(results_df))):
    print(f"\n[항목 {idx+1}]")
    print(f"질문: {eval_data['question'][idx][:100]}...")
    print(f"답변: {eval_data['answer'][idx][:100]}...")

    # 메트릭 점수 출력
    row = results_df.iloc[idx]

    if 'faithfulness' in results_df.columns and pd.notna(row['faithfulness']):
        print(f"   📊 Faithfulness: {row['faithfulness']:.3f}")

    if 'answer_relevancy' in results_df.columns and pd.notna(row['answer_relevancy']):
        print(f"   📊 Answer Relevancy: {row['answer_relevancy']:.3f}")

    if 'context_precision' in results_df.columns and pd.notna(row['context_precision']):
        print(f"   📊 Context Precision: {row['context_precision']:.3f}")

    if 'context_recall' in results_df.columns and pd.notna(row['context_recall']):
        print(f"   📊 Context Recall: {row['context_recall']:.3f}")

    print("   " + "-"*70)

# 낮은 점수 항목 분석
print("\n" + "="*80)
print("⚠️ 개선이 필요한 항목")
print("="*80)

if 'faithfulness' in results_df.columns:
    # NaN이 아닌 값만 필터링
    valid_faith = results_df['faithfulness'].notna()
    low_faith = results_df[valid_faith & (results_df['faithfulness'] < 0.7)]

    if len(low_faith) > 0:
        print(f"\n📌 Faithfulness < 0.7: {len(low_faith)}개")
        for i, (idx, row) in enumerate(low_faith.head(3).iterrows()):
            print(f"   {i+1}. [{row['faithfulness']:.3f}] {eval_data['question'][idx][:60]}...")
    else:
        print("\n✅ Faithfulness: 모든 항목이 0.7 이상!")

if 'answer_relevancy' in results_df.columns:
    valid_rel = results_df['answer_relevancy'].notna()
    low_rel = results_df[valid_rel & (results_df['answer_relevancy'] < 0.7)]

    if len(low_rel) > 0:
        print(f"\n📌 Answer Relevancy < 0.7: {len(low_rel)}개")
        for i, (idx, row) in enumerate(low_rel.head(3).iterrows()):
            print(f"   {i+1}. [{row['answer_relevancy']:.3f}] {eval_data['question'][idx][:60]}...")
    else:
        # NaN 개수 확인
        nan_count = (~valid_rel).sum()
        if nan_count > 0:
            print(f"\n⚠️ Answer Relevancy: {nan_count}개 항목 평가 실패 (Embeddings 오류)")
        else:
            print("\n✅ Answer Relevancy: 모든 항목이 0.7 이상!")

# -------------------
# 결과 저장
# -------------------
# 원본 데이터와 결과 병합
combined_results = []
for i in range(len(eval_data['question'])):
    result_dict = {
        '질문': eval_data['question'][i],
        '답변': eval_data['answer'][i],
        '근거_개수': len(eval_data['contexts'][i])
    }

    # 메트릭 점수 추가
    row = results_df.iloc[i]
    for metric in ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']:
        if metric in results_df.columns:
            result_dict[metric] = row[metric] if pd.notna(row[metric]) else None
        else:
            result_dict[metric] = None

    combined_results.append(result_dict)

combined_df = pd.DataFrame(combined_results)

# 저장
output_file = '/content/ragas_evaluation_results.csv'
combined_df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n✅ 결과 저장 완료: {output_file}")

# 통계 요약
print("\n" + "="*80)
print("📊 평가 통계")
print("="*80)
print(f"총 항목 수: {len(combined_df)}")
for metric in ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']:
    if metric in combined_df.columns:
        valid_count = combined_df[metric].notna().sum()
        if valid_count > 0:
            avg = combined_df[metric].mean()
            print(f"{metric}: 평균 {avg:.4f} (유효: {valid_count}/{len(combined_df)})")
        else:
            print(f"{metric}: 모든 항목 평가 실패")

# 다운로드 (Colab용)
from google.colab import files
files.download(output_file)
print("\n✅ 파일 다운로드 완료!")